## 🧠 RAG MEMORY MAP
**Remember:** Load → Embed → Load Documents → Chunk → Combine → Store → Retrieve → Hybrid Search → Augment → Generate → Answer

# Building an END-TO-END RAG Chain

## 🔐 STEP 1 — ENVIRONMENT / CONFIG
**Keyword:** `Environment Setup`

**Remember:** Load `.env` → make configuration/API settings available.

In [1]:

import os
from dotenv import load_dotenv

load_dotenv(".env")

True

## 🧩 STEP 2 — EMBEDDINGS
**Keyword:** `Embeddings = Text → Vectors`

**Remember:** Convert text into numerical vectors so semantic similarity can be searched.

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 📥 STEP 3 — LOAD WIKIPEDIA DATA
**Keyword:** `Document Loading`

**Remember:** JSON/JSONL → LangChain `Document` objects.

In [3]:
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(
    file_path=os.getenv("JSON_FILE_PATH"),
    jq_schema='.',
    text_content=False,
    json_lines=True
)

wiki_docs = loader.load()
print(len(wiki_docs))

1801


## 🧹 STEP 4 — CLEAN / PREPARE WIKIPEDIA DOCUMENTS
**Keyword:** `Preprocessing`

**Remember:** Raw JSON → extract `title`, `id`, `paragraphs` → create clean `Document` objects with metadata.

In [4]:
import json
from langchain_core.documents import Document

wiki_docs_processed = []

for doc in wiki_docs:

    # Convert JSON string → Python dictionary
    data = json.loads(doc.page_content)

    # Metadata
    metadata = {
        "title": data["title"],
        "id": data["id"],
        "source": "wikipedia"
    }

    # Combine paragraphs
    text = " ".join(data["paragraphs"])

    # Create LangChain Document
    wiki_docs_processed.append(
        Document(
            page_content=text,
            metadata=metadata
        )
    )


print("Processed Wikipedia documents:", len(wiki_docs_processed))

Processed Wikipedia documents: 1801


## ✂️ STEP 5 — SEMANTIC CHUNKING
**Keyword:** `Semantic Chunking`

**Remember:** Split documents by **meaning**, not just fixed character count.

In [5]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile"
)

wiki_chunks = semantic_splitter.split_documents(
    wiki_docs_processed
)

print("Wikipedia semantic chunks:", len(wiki_chunks))

Wikipedia semantic chunks: 3613


## 📄 STEP 6 — PDF LOADING + CHUNKING FUNCTION
**Keyword:** `PDF → Pages → Semantic Chunks`

**Remember:** Load PDF pages → apply semantic chunking → return chunks.

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader # Document Loader
from langchain.text_splitter import RecursiveCharacterTextSplitter # Chunking Strategy

from langchain_huggingface import HuggingFaceEmbeddings

## 📝 ALTERNATIVE — SIMPLE CHUNKING
**Keyword:** `Fixed-Size Chunking`

**Remember:** Character-based chunking is an alternative to semantic chunking.

In [7]:
# def create_simple_chunks(file_path, chunk_size=500, chunk_overlap=40):
#   print("Loading pages: ", file_path)
#   loader = PyMuPDFLoader(file_path)
#   doc_pages = loader.load()

#   print("Chunking pages :", file_path)
#   splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
#   doc_chunks = splitter.split_documents(doc_pages)

#   print("Finishing Loading and Processing steps with PDF Dataset :", file_path)
#   print("******************************************")
#   return doc_chunks

def create_semantic_chunks(file_path, embeddings):
    print("Loading pages:", file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print("Semantic chunking:", file_path)
    splitter = SemanticChunker(
        embeddings=embeddings,
        breakpoint_threshold_type="percentile"
    )

    doc_chunks = splitter.split_documents(doc_pages)

    print("Finished:", file_path)
    print("******************************************")

    return doc_chunks



## 📂 STEP 7 — FIND PDF FILES
**Keyword:** `File Discovery`

**Remember:** Find all PDFs inside the `content/` folder.

In [8]:
from glob import glob

## 📚 STEP 8 — PDF FILE LIST
**Keyword:** `Input Files`

**Remember:** Check which PDF files will be processed.

In [9]:
pdf_files = glob('content/*.pdf') 
pdf_files

['content\\attention_paper.pdf',
 'content\\cnn_paper.pdf',
 'content\\resnet_paper.pdf',
 'content\\vision_transformer.pdf']

## 🔄 STEP 9 — PROCESS ALL PDFs
**Keyword:** `Batch Processing`

**Remember:** Loop through every PDF → semantic chunk each → combine into `paper_docs`.

In [10]:
paper_docs = []

for pdf_file in pdf_files:
    paper_docs.extend(create_semantic_chunks(file_path=pdf_file,embeddings=embeddings))

Loading pages: content\attention_paper.pdf
Semantic chunking: content\attention_paper.pdf
Finished: content\attention_paper.pdf
******************************************
Loading pages: content\cnn_paper.pdf
Semantic chunking: content\cnn_paper.pdf
Finished: content\cnn_paper.pdf
******************************************
Loading pages: content\resnet_paper.pdf
Semantic chunking: content\resnet_paper.pdf
Finished: content\resnet_paper.pdf
******************************************
Loading pages: content\vision_transformer.pdf
Semantic chunking: content\vision_transformer.pdf
Finished: content\vision_transformer.pdf
******************************************


## 🔢 STEP 10 — CHECK PDF CHUNK COUNT
**Keyword:** `Validation`

**Remember:** Verify how many PDF chunks were created.

In [11]:
print(len(paper_docs))

190


## 🔍 STEP 11 — INSPECT A CHUNK
**Keyword:** `Data Inspection`

**Remember:** Look at one generated chunk to verify content + metadata.

In [12]:
paper_docs[0]

Document(metadata={'source': 'content\\attention_paper.pdf', 'file_path': 'content\\attention_paper.pdf', 'page': 0, 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'creationDate': 'D:20230803000729Z', 'modDate': 'D:20230803000729Z', 'trapped': ''}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works. Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstra

## 🧺 STEP 12 — COMBINE ALL CHUNKS
**Keyword:** `Unified Corpus`

**Remember:** `wiki_chunks + paper_docs` → one collection: `total_docs`.

In [13]:
total_docs = wiki_chunks + paper_docs

print("========================================")
print("Total chunks:", len(total_docs))
print("========================================")

Total chunks: 3803


## 🗄️ STEP 13 — CHROMA VECTOR DATABASE
**Keyword:** `Vector Store`

**Remember:** Store document embeddings in Chroma for fast similarity search.

In [14]:
from langchain_chroma import Chroma

## 💾 STEP 14 — CREATE / PERSIST VECTOR DB
**Keyword:** `Index Once → Reuse`

**Remember:** Create the Chroma database from documents + embeddings and persist it.

In [15]:
# Create vector DB of docs and embedding
## Please note below steps use / execute only one time

chroma_db = Chroma.from_documents(
    documents=total_docs,
    embedding=embeddings,
    persist_directory="./my_db",
    collection_metadata={
        "hnsw:space": "cosine"
    }
)
# HNSW - Hierarchical Navigable Small World

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## 🔌 STEP 15 — USE VECTOR DB
**Keyword:** `Load / Access Vector Store`

**Remember:** Work with the existing Chroma database.

In [16]:
chroma_db

## 🔎 STEP 16 — DENSE + SPARSE RETRIEVERS
**Keyword:** `Retrieval`

**Remember:** Vector search = **semantic meaning**; BM25 = **keyword matching**.

In [17]:
from langchain_community.retrievers import BM25Retriever

# Dense/vector retriever
vector_retriever = chroma_db.as_retriever(search_type="similarity",search_kwargs={"k": 5})

# Sparse/keyword retriever
bm25_retriever = BM25Retriever.from_documents(total_docs)
bm25_retriever.k = 5

## 🔀 STEP 17 — HYBRID RETRIEVAL
**Keyword:** `Vector + BM25 → Hybrid`

**Remember:** Combine dense vector retrieval and sparse BM25 retrieval using `EnsembleRetriever`.

In [18]:
from langchain.retrievers import EnsembleRetriever
# Hybrid retrieval
hybrid_retriever = EnsembleRetriever(
    retrievers=[
        vector_retriever,
        bm25_retriever
    ],
    weights=[0.5, 0.5]
)

## 👀 STEP 18 — DISPLAY RETRIEVED DOCUMENTS
**Keyword:** `Inspection / Debugging`

**Remember:** Show retrieved chunks + metadata to check whether retrieval is working.

In [19]:
from IPython.display import display, Markdown


def display_docs(docs):

    for i, doc in enumerate(docs, 1):

        print(f"\n========== Document {i} ==========")

        print("Metadata:")
        print(doc.metadata)

        print("\nContent Brief:")

        display(
            Markdown(
                doc.page_content[:4000]
            )
        )

        print("================================")

## ❓ STEP 19 — TEST QUERIES
**Keyword:** `Retrieval Testing`

**Remember:** Ask sample questions to test the retriever.

In [20]:
# Test queries

query1 = "what is self attention model?"

## 🎯 STEP 20 — RUN HYBRID RETRIEVAL
**Keyword:** `Query → Top-K Context`

**Remember:** User query → hybrid retriever → relevant documents → display results.

In [21]:
# Hybrid retrieval

top_docs = hybrid_retriever.invoke(query1)

# Display retrieved documents

display_docs(top_docs)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



========== Document 1 ==========
Metadata:
{'author': '', 'creationDate': 'D:20230803000729Z', 'creator': 'LaTeX with hyperref', 'file_path': 'content\\attention_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20230803000729Z', 'page': 5, 'producer': 'pdfTeX-1.40.25', 'source': 'content\\attention_paper.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}

Content Brief:


We chose the sinusoidal version
because it may allow the model to extrapolate to sequence lengths longer than the ones encountered
during training. 4
Why Self-Attention
In this section we compare various aspects of self-attention layers to the recurrent and convolu-
tional layers commonly used for mapping one variable-length sequence of symbol representations
(x1, ..., xn) to another sequence of equal length (z1, ..., zn), with xi, zi ∈Rd, such as a hidden
layer in a typical sequence transduction encoder or decoder. Motivating our use of self-attention we
consider three desiderata. One is the total computational complexity per layer.


========== Document 2 ==========
Metadata:
{'source': 'content\\attention_paper.pdf', 'file_path': 'content\\attention_paper.pdf', 'page': 13, 'total_pages': 15, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'creationDate': 'D:20230803000729Z', 'modDate': 'D:20230803000729Z', 'trapped': ''}

Content Brief:


The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
. <EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
. <EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
. <EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
. <EOS>
<pad>
Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top:
Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5
and 6. Note that the attentions are very sharp for this word.

...

Scaled Dot-Product Attention
Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel. of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key. 3.2.1
Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the
query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the
values. In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrix Q. The keys and values are also packed together into matrices K and V . We compute
the matrix of outputs as:
Attention(Q, K, V ) = softmax(QKT
√dk
)V
(1)
The two most commonly used attention functions are additive attention [2], and dot-product (multi-
plicative) attention. Dot-product attention is identical to our algorithm, except for the scaling factor
of
1
√dk . Additive attention computes the compatibility function using a feed-forward network with
a single hidden layer. While the two are similar in theoretical complexity, dot-product attention is
much faster and more space-efficient in practice, since it can be implemented using highly optimized
matrix multiplication code. While for small values of dk the two mechanisms perform similarly, additive attention outperforms
dot product attention without scaling for larger values of dk [3]. We suspect that for large values of
dk, the dot products grow large in magnitude, pushing the softmax function into regions where it has
extremely small gradients 4. To counteract this effect, we scale the dot products by
1
√dk . 3.2.2
Multi-Head Attention
Instead of performing a single attention function with dmodel-dimensional keys, values and queries,
we found it beneficial to linearly project the queries, keys and values h times with different, learned
linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional
4To illustrate why the dot products get large, assume that the components of q and k are independent random
variables with mean 0 and variance 1. Then their dot product, q · k = Pdk
i=1 qiki, has mean 0 and variance dk.


========== Document 9 ==========
Metadata:
{'author': '', 'creationDate': 'D:20230803000729Z', 'creator': 'LaTeX with hyperref', 'file_path': 'content\\attention_paper.pdf', 'format': 'PDF 1.5', 'keywords': '', 'modDate': 'D:20230803000729Z', 'page': 2, 'producer': 'pdfTeX-1.40.25', 'source': 'content\\attention_paper.pdf', 'subject': '', 'title': '', 'total_pages': 15, 'trapped': ''}

Content Brief:


The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer
itself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding
layers, produce outputs of dimension dmodel = 512. Decoder:
The decoder is also composed of a stack of N = 6 identical layers. In addition to the two
sub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head
attention over the output of the encoder stack. Similar to the encoder, we employ residual connections
around each of the sub-layers, followed by layer normalization. We also modify the self-attention
sub-layer in the decoder stack to prevent positions from attending to subsequent positions. This
masking, combined with fact that the output embeddings are offset by one position, ensures that the
predictions for position i can depend only on the known outputs at positions less than i. 3.2
Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3



========== Document 10 ==========
Metadata:
{'source': 'content\\vision_transformer.pdf', 'file_path': 'content\\vision_transformer.pdf', 'page': 19, 'total_pages': 22, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.21', 'creationDate': 'D:20210604001958Z', 'modDate': 'D:20210604001958Z', 'trapped': ''}

Content Brief:


Published as a conference paper at ICLR 2021
102
Total compute [exaFLOPs]
0.500
0.525
0.550
0.575
0.600
0.625
0.650
ImageNet 5-shot linear top-1 accuracy
AxialViT-B/16
AxialViT-B/32
ViT-B/16
ViT-B/32
ResNet50
AxialResNet50
102
103
Peak inference speed [img/sec/core]
0.500
0.525
0.550
0.575
0.600
0.625
0.650
ImageNet 5-shot linear top-1 accuracy
AxialViT-B/16
AxialViT-B/32
ViT-B/16
ViT-B/32
ResNet50
AxialResNet50
Figure 13: Performance of Axial-Attention based models, in terms of top-1 accuracy on ImageNet
5-shot linear, versus their speed in terms of number of FLOPs (left) and inference time (left). the cost of more compute. This is because in Axial-ViT models, each Transformer block with global
self-attention is replaced by two Axial Transformer blocks, one with row and one with column self-
attention and although the sequence length that self-attention operates on is smaller in axial case,
there is a extra MLP per Axial-ViT block. For the AxialResNet, although it looks reasonable in
terms of accuracy/compute trade-off (Figure 13, left), the naive implementation is extremely slow
on TPUs (Figure 13, right). D.7
ATTENTION DISTANCE
To understand how ViT uses self-attention to integrate information across the image, we analyzed
the average distance spanned by attention weights at different layers (Figure 11). This “attention
distance” is analogous to receptive ﬁeld size in CNNs. Average attention distance is highly variable
across heads in lower layers, with some heads attending to much of the image, while others attend
to small regions at or near the query location. As depth increases, attention distance increases for all
heads. In the second half of the network, most heads attend widely across tokens. D.8
ATTENTION MAPS
To compute maps of the attention from the output token to the input space (Figures 6 and 14), we
used Attention Rollout (Abnar & Zuidema, 2020). Brieﬂy, we averaged attention weights of ViT-
L/16 across all heads and then recursively multiplied the weight matrices of all layers. This accounts
for the mixing of attention across tokens through all layers. D.9
OBJECTNET RESULTS
We also evaluate our ﬂagship ViT-H/14 model on the ObjectNet benchmark following the evaluation
setup in Kolesnikov et al. (2020), resulting in 82.1% top-5 accuracy and 61.7% top-1 accuracy.

## 🧠 STEP 21 — AUGMENTATION / PROMPT
**Keyword:** `Question + Retrieved Context → Prompt`

**Remember:** Build the RAG prompt that tells the LLM to answer using retrieved context.

In [22]:
# Augmentation method - Enhanced Prompt

from langchain_core.prompts import ChatPromptTemplate


rag_prompt = """You are an assistant who is an expert in question-answering tasks.
                Your rule here is to answer the following question using only the following pieces of
                retrieval context. If the answer is not in the context, do not make up answers, just say that I don't know.
                Keep the answer detailed and well formatted based on the information from the context.
                You can also share the citation of the response as from where this response comes from.

                Question:
                {question}

                Context:
                {context}

                Answer:
                """
rag_prompt_template = ChatPromptTemplate.from_template(rag_prompt)

## ⚙️ STEP 22 — BUILD RAG CHAIN
**Keyword:** `Retrieve → Format → Prompt → LLM`

**Remember:** `hybrid_retriever` gets context → `format_docs` formats it → `RunnablePassthrough` keeps the question → prompt combines both → Ollama LLM generates.

In [24]:
from langchain_core.runnables import RunnablePassthrough
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)


qa_rag_chain = (
    {"context":(hybrid_retriever  | format_docs),
     "question": RunnablePassthrough()}
    | rag_prompt_template
    | llm
)


## 🚀 STEP 23 — RUN END-TO-END RAG
**Keyword:** `Question → Answer`

**Remember:** Invoke the complete chain → retrieve → augment → generate → display the final answer.

In [26]:
query = "What is Machine Learning?"

result = qa_rag_chain.invoke(query)

display(Markdown(result.content))

**Definition of Machine Learning**

Machine learning is a subfield of computer science that gives computers the ability to learn without being explicitly programmed (Arthur Samuel, 1959). It explores the study and construction of algorithms that can learn and make predictions on data. These algorithms follow programmed instructions but can also make predictions or decisions based on data. They build a model from sample inputs.

**Key Characteristics**

Machine learning is used in situations where designing and programming explicit algorithms cannot be done (e.g., spam filtering, detection of network intruders, optical character recognition, search engines, and computer vision). The changes that affect the amount of next period consumption are unpredictable or not given information, making it impossible to expect the changes.

**Types of Machine Learning**

There are three types of machine learning:

1. **Supervised Learning**: This task involves inferring a function from labelled training data (e.g., predicting house prices based on features like number of bedrooms and square footage).
2. **Unsupervised Learning**: In this type, the algorithm learns patterns or relationships in the data without any prior knowledge of the correct output.
3. **Semi-Supervised Learning**: This approach combines supervised and unsupervised learning, where the algorithm is trained on both labelled and unlabelled data.

**Deep Learning**

Deep learning (also known as deep structured learning or hierarchical learning) is a type of machine learning that uses neural networks to make predictions. It is mostly used with certain kinds of neural networks and can be supervised, unsupervised, or semi-supervised.

**Citation**
Arthur Samuel, "A Computer Program for Playing Checkers" (1959)

Note: The provided context does not mention the term "machine learning" explicitly until Arthur Samuel coined it in 1959. However, based on the information provided, we can infer that machine learning is a subfield of computer science that enables computers to learn from data without explicit programming.